Kugelwolkenmodell-Mathematik. Erstellt JSON Objekt.

In [11]:
import numpy as np
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List

app = FastAPI(title="Kugelwolkenmodell Backend-API")

# CORS erlauben, damit auch lokale Webseiten (Three.js) auf den Server zugreifen dürfen
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ==========================================
# MATHEMATISCHER KERN (Aus den vorherigen Schritten)
# ==========================================

import numpy as np

class Kugelwolke:
    def __init__(self, id_num, elektronen, richtung_lokal):
        self.id = id_num
        self.elektronen = elektronen  # 1 = reaktiv (blau), 2 = frei (grau), 3 = bindung (grün)
        self.richtung_lokal = richtung_lokal  # 3D-Vektor relativ zum Kern

class Atom:
    def __init__(self, uid, symbol, position=[0.0, 0.0, 0.0]):
        self.uid = uid
        self.symbol = symbol
        self.position = np.array(position, dtype=float)
        self.rotation = np.eye(3)  # Startet unrotiert (Einheitsmatrix)
        self.wolken = []

        self._baue_wolken()

    def _baue_wolken(self):
        if self.symbol == "H":
            # Wasserstoff hat nur 1 Wolke direkt im Zentrum [0,0,0]
            self.wolken.append(Kugelwolke(id_num=0, elektronen=1, richtung_lokal=[0.0, 0.0, 0.0]))

        elif self.symbol == "O":
            # Sauerstoff: 4 Wolken im Tetraederwinkel (Abstand 0.6)
            t_coords = [
                [ 1,  1,  1],
                [-1, -1,  1],
                [-1,  1, -1],
                [ 1, -1, -1]
            ]
            vektoren = [np.array(c) / np.linalg.norm(c) * 0.6 for c in t_coords]

            # 6 Valenzelektronen: 2 freie Paare (je 2e-), 2 reaktive Wolken (je 1e-)
            self.wolken.append(Kugelwolke(0, elektronen=2, richtung_lokal=vektoren[0].tolist())) # frei
            self.wolken.append(Kugelwolke(1, elektronen=2, richtung_lokal=vektoren[1].tolist())) # frei
            self.wolken.append(Kugelwolke(2, elektronen=1, richtung_lokal=vektoren[2].tolist())) # reaktiv
            self.wolken.append(Kugelwolke(3, elektronen=1, richtung_lokal=vektoren[3].tolist())) # reaktiv

    def to_dict(self):
        return {
            "uid": self.uid,
            "symbol": self.symbol,
            "position": self.position.tolist(),
            "rotation": self.rotation.flatten().tolist(),
            "wolken": [w.__dict__ for w in self.wolken]
        }

# ==========================================
# DER NEUE WELT-ZUSTAND: EIN SAUERSTOFF, ZWEI WASSERSTOFFE
# ==========================================
def initialisiere_welt():
    return {
        "O_Zentrum": Atom("O_Zentrum", "O", position=[0.0, 0.0, 0.0]),
        "H_links":   Atom("H_links", "H", position=[-2.5, 1.5, 0.5]),  # wild im Raum verteilt
        "H_rechts":  Atom("H_rechts", "H", position=[2.5, -1.5, 0.5])
    }

WELT_ZUSTAND = initialisiere_welt()




# Procrustes & Bindungslogik
def berechne_procrustes_rotation(A, B):
    M = A @ B.T
    U, S, Vt = np.linalg.svd(M)
    R = U @ Vt
    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt
    return R



# ==========================================
# API-ENDPOINTS (DIE BRÜCKE ZUR UI)
# ==========================================

@app.get("/status")
def get_status():
    return {uid: atom.to_dict() for uid, atom in WELT_ZUSTAND.items()}

@app.post("/reset")
def reset_simulation():
    global WELT_ZUSTAND
    WELT_ZUSTAND = initialisiere_welt()
    return {"status": "success", "message": "Wasser-Setup bereit."}



class BindungsAnfrage(BaseModel):
    atom_A_id: str
    wolken_A_ids: List[int]
    atom_B_id: str
    wolken_B_ids: List[int]

@app.post("/binde")
def binde_atome(daten: dict):
    atom_A_id = daten["atom_A_id"]
    wolken_A_ids = daten["wolken_A_ids"]
    atom_B_id = daten["atom_B_id"]
    wolken_B_ids = daten["wolken_B_ids"]

    atom_A = WELT_ZUSTAND[atom_A_id]
    atom_B = WELT_ZUSTAND[atom_B_id]

    # 1. Lokale Vektoren extrahieren
    punkte_A_lokal = [np.array(atom_A.wolken[i].richtung_lokal) for i in wolken_A_ids]
    punkte_B_lokal = [np.array(atom_B.wolken[i].richtung_lokal) for i in wolken_B_ids]

    # 2. Symmetrische Wasserstoff-Korrektur: Wer ist H und wer ist das Zentrum (O)?
    # Wir wollen immer, dass das H-Atom auf den Abstand von 1.0 Einheiten entlang der O-Wolkenachse gesetzt wird.
    if atom_B.symbol == "H" and atom_A.symbol == "O":
        for i, p in enumerate(punkte_B_lokal):
            richtung_sauerstoff_wolke = punkte_A_lokal[i] / np.linalg.norm(punkte_A_lokal[i])
            punkte_B_lokal[i] = richtung_sauerstoff_wolke * -1.0

    elif atom_A.symbol == "H" and atom_B.symbol == "O":
        for i, p in enumerate(punkte_A_lokal):
            richtung_sauerstoff_wolke = punkte_B_lokal[i] / np.linalg.norm(punkte_B_lokal[i])
            punkte_A_lokal[i] = richtung_sauerstoff_wolke * -1.0

    # 3. Globale Zielpunkte für die Translation berechnen
    punkte_A_global = [atom_A.position + atom_A.rotation @ p for p in punkte_A_lokal]

    # 4. Verschiebung (Translation) berechnen
    zentrum_A = np.mean(punkte_A_global, axis=0)
    zentrum_B = np.mean(punkte_B_lokal, axis=0)

    # Wir bewegen immer das Atom B zum Atom A
    atom_B.position = zentrum_A - atom_B.rotation @ zentrum_B

    # 5. Status auf "Gebunden" (3) setzen
    for i in wolken_A_ids: atom_A.wolken[i].elektronen = 3
    for i in wolken_B_ids: atom_B.wolken[i].elektronen = 3

    return {"status": "success"}



Server starten

In [12]:
import multiprocessing
import uvicorn
import nest_asyncio

# Erlaubt verschachtelte Event-Loops in Jupyter/Colab Notebooks
nest_asyncio.apply()

def start_api():
    # Wir übergeben direkt das 'app'-Objekt aus Zelle 1
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Server in einen separaten Hintergrundprozess auslagern
server_prozess = multiprocessing.Process(target=start_api)
server_prozess.start()
print("Backend-Server wurde erfolgreich im Hintergrund auf Port 8000 gestartet!")

Backend-Server wurde erfolgreich im Hintergrund auf Port 8000 gestartet!


Test server

In [13]:
#server_prozess.terminate()
#print("Server gestoppt.")

In [14]:
# 1. Installiere pyngrok
!pip install pyngrok

from pyngrok import ngrok

# ==========================================
# HIER FÜGST DU DEINEN AUTHTOKEN EIN
# ==========================================
# Ersetze 'DEIN_NGROK_AUTHTOKEN_HIER' mit dem langen Code von der ngrok-Webseite
NGROK_TOKEN = "1Y1jBALsPqgA8RQOJd4HDib3Ofh_6HgoK1dPV8iBcKbVvxbQb"

# Dem ngrok-System den Token mitteilen
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Jetzt erst den Tunnel zu Port 8000 öffnen
public_url = ngrok.connect(8000, "http")
print("Deine öffentliche API-Zustandsadresse für deine lokale UI lautet:")
print(f"{public_url.public_url}/status")

Deine öffentliche API-Zustandsadresse für deine lokale UI lautet:
https://79e4-34-132-190-178.ngrok-free.app/status


In [15]:
from fastapi.responses import HTMLResponse

@app.get("/", response_class=HTMLResponse)
def read_root():
    # 1. Lies die lokale index.html ein
    with open("index.html", "r", encoding="utf-8") as f:
        html_content = f.read()

    # 2. Wir suchen NUR nach dem Variablen-Namen und überschreiben die Zuweisung komplett.
    # Egal ob in deiner index.html 'http...' oder 'https...' steht:
    # Wir zwingen das JavaScript dazu, die aktuelle Server-Adresse dynamisch zu ermitteln.

    # Wir ersetzen die alte Zeile komplett mit einer sauberen JavaScript-Anweisung:
    html_content = html_content.replace(
        "const API_URL = 'http://127.0.0.1:8000';",
        "const API_URL = window.location.origin;"
    )

    # Falls du in der index.html schon die https-ngrok-Adresse fest eingetragen hattest,
    # fangen wir das hier sicherheitshalber auch noch ab:
    if "const API_URL = 'https://" in html_content:
        # Wir suchen das Ende der Zeile und ersetzen sie sauber
        start_idx = html_content.find("const API_URL = 'https://")
        end_idx = html_content.find("';", start_idx) + 2
        alte_zeile = html_content[start_idx:end_idx]
        html_content = html_content.replace(alte_zeile, "const API_URL = window.location.origin;")

    return html_content